# E8b: variantes de XGBoost

Criterio fijado antes en `README.md` (E8b): AUC = mediana por persona dejando una persona fuera, media de 5 semillas; una variante se adopta si supera a la referencia por 0.01 o más.

1. Referencia: XGBoost único con MPUD.
2. Dos modelos (F y S), combinados como 1 − (1 − p_F)(1 − p_S).
3. Poda de los bloques de 60 min y de hora y contexto.
4. *d* = 4 con la mejor variante, en los ejemplos comunes.

In [1]:
import os, json
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
import numpy as np
import pandas as pd
from experiments.modeling import load, feature_cols
from experiments import variants as V
SEEDS = [0, 1, 2, 3, 4]
real, mpud = load("real"), load("mpud")
cols = feature_cols(real)
print(len(cols), "features;", len(V.pruned(cols)), "after pruning")

52 features; 36 after pruning


## Variantes 1–3: todas las features y podadas

In [2]:
full = V.run(real, mpud, cols, SEEDS)
prun = V.run(real, mpud, V.pruned(cols), SEEDS)
table = pd.concat({"all features": full.mean(), "pruned": prun.mean()}, axis=1).round(3)
table

,all features,pruned
single,0.585,0.594
two_models,0.574,0.574
two_models_max,0.574,0.575
f_model_on_y_f,0.705,0.693
single_on_y_f,0.538,0.538
s_model_on_y_s,0.689,0.689
single_on_y_s,0.558,0.562


In [3]:
spread = pd.concat({"all features": full.agg(["min", "max"]).T, "pruned": prun.agg(["min", "max"]).T}, axis=1).round(3)
spread

all features        pruned       
                        min    max    min    max
single                0.571  0.598  0.577  0.604
two_models            0.560  0.585  0.558  0.585
two_models_max        0.567  0.581  0.557  0.582
f_model_on_y_f        0.699  0.711  0.689  0.697
single_on_y_f         0.533  0.544  0.518  0.551
s_model_on_y_s        0.681  0.699  0.679  0.699
single_on_y_s         0.551  0.567  0.549  0.569

## Decisión de las variantes 1–3

In [4]:
ref = table.loc["single", "all features"]
candidates = {("single", "pruned"): table.loc["single", "pruned"],
              ("two_models", "all features"): table.loc["two_models", "all features"],
              ("two_models", "pruned"): table.loc["two_models", "pruned"]}
for k, v in candidates.items():
    print(f"{k}: {v:.3f} ({v - ref:+.3f} vs reference {ref:.3f}) -> adopt: {v - ref >= 0.01}")
adopted = [k for k, v in candidates.items() if v - ref >= 0.01]
best = max(adopted, key=candidates.get) if adopted else ("single", "all features")
print("best variant:", best)

('single', 'pruned'): 0.594 (+0.009 vs reference 0.585) -> adopt: False
('two_models', 'all features'): 0.574 (-0.011 vs reference 0.585) -> adopt: False
('two_models', 'pruned'): 0.574 (-0.011 vs reference 0.585) -> adopt: False
best variant: ('single', 'all features')


## Variante 4: *d* = 4

La mejor variante con *d* = 3 y con *d* = 4, evaluadas en los mismos ejemplos.

In [5]:
real4 = pd.read_csv("data/processed/real_features_d4.csv", parse_dates=["day"])
mpud4 = pd.read_csv("data/processed/mpud_features_d4.csv", parse_dates=["day"])
kind, feats = best
use = cols if feats == "all features" else V.pruned(cols)
common = V.common_rows(real, real4)
rows = []
for seed in SEEDS:
    V.set_seed(seed)
    if kind == "single":
        s3, s4 = V.scores_for(real, mpud, use, "y"), V.scores_for(real4, mpud4, use, "y")
    else:
        s3 = 1 - (1 - V.scores_for(real, mpud, use, "y_f")) * (1 - V.scores_for(real, mpud, use, "y_s"))
        s4 = 1 - (1 - V.scores_for(real4, mpud4, use, "y_f")) * (1 - V.scores_for(real4, mpud4, use, "y_s"))
    order = real[common].set_index(["user_id", "t_utc"]).index
    s4_aligned = pd.Series(s4, index=real4.set_index(["user_id", "t_utc"]).index).loc[order].to_numpy()
    rows.append({"seed": seed, "d3_on_common": V.auc(real[common], s3[common]),
                 "d4_on_common": V.auc(real4.set_index(["user_id", "t_utc"]).loc[order].reset_index(), s4_aligned),
                 "d4_all": V.auc(real4, s4)})
dres = pd.DataFrame(rows).set_index("seed")
print(f"common examples: {common.sum()} of {len(real)}")
dres.agg(["mean", "min", "max"]).round(3)

common examples: 14758 of 16281


,d3_on_common,d4_on_common,d4_all
mean,0.582,0.585,0.585
min,0.572,0.581,0.581
max,0.595,0.589,0.589


In [6]:
gain_d4 = dres["d4_on_common"].mean() - dres["d3_on_common"].mean()
print(f"d = 4 vs d = 3 on common examples: {gain_d4:+.3f} -> adopt d = 4: {gain_d4 >= 0.01}")

d = 4 vs d = 3 on common examples: +0.004 -> adopt d = 4: False


## Guardar

In [7]:
out = {"variants_mean": table.to_dict(), "variants_by_seed": {"all": full.to_dict(), "pruned": prun.to_dict()},
       "best": list(best), "d4": dres.to_dict(), "d4_gain": gain_d4}
Path("data/reports/e8b_xgb_variants.json").write_text(json.dumps(out, indent=2, default=float))
print("-> data/reports/e8b_xgb_variants.json")

-> data/reports/e8b_xgb_variants.json
